<a href="https://colab.research.google.com/github/iamuhd/my_ML_Intership_at_FLYRANK-AI/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

This notebook builds a transparent, rule-based content-refresh queue. The CSV queue is regenerated locally and ignored by git; compact JSON receipts are written for reproducibility.


## 1. My rule and its reason codes

A page moves up the queue when it has visible demand and a practical opportunity: it is old, has a weak search position, has low CTR for its visibility, or is unusually short. The two signal checks are **visibility × freshness** and **visibility × search opportunity**.

Reason codes: `stale_visible_page`, `position_opportunity`, `low_ctr_visible_page`, `thin_visible_page`, and `general_review`. The score excludes `trend_direction`, `trend_pct`, `is_declining_label`, and identifiers; those are reserved for evaluation or grouping.

In [6]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

local_paths = [
    (Path('data/raw/content_refresh_anonymized.csv'), Path('.')),
    (Path('../../data/raw/content_refresh_anonymized.csv'), Path('../..')),
]
local_match = next(((data, root) for data, root in local_paths if data.exists()), None)
if local_match is None:
    data_path = 'https://raw.githubusercontent.com/iamuhd/my_ML_Intership_at_FLYRANK-AI/main/data/raw/content_refresh_anonymized.csv'
    repo_root = Path('.')
else:
    data_path, repo_root = local_match

raw = pd.read_csv(data_path)
required = ['content_id', 'client_id', 'impressions_90d', 'content_age_days', 'avg_position', 'ctr', 'days_since_last_update', 'word_count', 'trend_direction']
missing = sorted(set(required) - set(raw.columns))
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df = raw[(raw['impressions_90d'] > 0) & (raw['content_age_days'] >= 90)].drop_duplicates('content_id').copy()
for column in ['impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'word_count', 'engagement_rate', 'sessions_90d']:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)

def percentile_rank(values):
    return pd.Series(values).rank(method='average', pct=True).fillna(0).to_numpy()

def minmax(values):
    values = pd.Series(values).astype(float).fillna(0)
    lo, hi = values.min(), values.max()
    return pd.Series(0.0, index=values.index) if hi == lo else (values - lo) / (hi - lo)

def reason_codes(row):
    reasons = []
    visible = row['impressions_90d'] >= 500
    if row['days_since_last_update'] >= 180 and visible:
        reasons.append('stale_visible_page')
    if visible and 0 < row['avg_position'] <= 20:
        reasons.append('position_opportunity')
    if visible and 0 < row['avg_position'] <= 20 and row['ctr'] < 0.5:
        reasons.append('low_ctr_visible_page')
    if 0 < row['word_count'] < 1200 and row['impressions_90d'] >= 250:
        reasons.append('thin_visible_page')
    return '|'.join(reasons) if reasons else 'general_review'

declining_rate = float(df['trend_direction'].eq('down').mean())
print(f'Rows screened: {len(df):,}')
print(f'Observed declining-label base rate: {declining_rate:.3f}')

Rows screened: 30,000
Observed declining-label base rate: 0.542


## 2. Build the ranked queue (writes the CSV and JSON receipts)

The CSV is a regenerated working artifact and remains ignored by git. The JSON files are compact run receipts and are intentionally suitable for committing.

In [7]:
visibility = pd.Series(percentile_rank(np.log1p(df['impressions_90d'])), index=df.index)
freshness = pd.Series(percentile_rank(df['days_since_last_update']), index=df.index)
position_quality = (1 - minmax(df['avg_position'].clip(lower=1, upper=50))) * (df['avg_position'] > 0).astype(int)
position_opportunity = position_quality * visibility
low_ctr_opportunity = (1 - minmax(df['ctr'].clip(lower=0, upper=df['ctr'].quantile(0.99)))) * visibility
depth_gap = (1 - pd.Series(percentile_rank(df['word_count']), index=df.index)) * visibility
df['baseline_action_score'] = (0.35 * visibility + 0.25 * freshness + 0.20 * position_opportunity + 0.15 * low_ctr_opportunity + 0.05 * depth_gap).clip(0, 1)
df['reason_codes'] = df.apply(reason_codes, axis=1)

def action(code):
    codes = set(str(code).split('|'))
    if 'thin_visible_page' in codes:
        return 'expand_and_refresh'
    if 'low_ctr_visible_page' in codes or 'position_opportunity' in codes:
        return 'refresh_and_review_ctr'
    if 'stale_visible_page' in codes:
        return 'refresh'
    return 'monitor'

df['suggested_action'] = df['reason_codes'].map(action)
df['is_declining_label'] = df['trend_direction'].eq('down').astype(int)
df = df.sort_values(['baseline_action_score', 'impressions_90d'], ascending=[False, False]).reset_index(drop=True)
df['baseline_rank'] = np.arange(1, len(df) + 1)

output_columns = ['content_id', 'client_id', 'baseline_rank', 'baseline_action_score', 'suggested_action', 'reason_codes', 'is_declining_label', 'trend_direction', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr', 'engagement_rate', 'content_age_days', 'days_since_last_update', 'word_count']
out = df[output_columns].copy()
output_dir = repo_root / 'work' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / 'baseline_action_score.csv'
metrics_path = output_dir / 'baseline_metrics.json'
review_path = output_dir / 'baseline_top10_review.json'
leakage_path = output_dir / 'baseline_leakage_check.json'
out.to_csv(csv_path, index=False)

def precision_at_k(frame, k):
    return float(frame.head(min(k, len(frame)))['is_declining_label'].mean())

base_rate = float(out['is_declining_label'].mean())
metrics = {
    'rows_screened': int(len(out)),
    'declining_label_base_rate': base_rate,
    'precision_at_10': precision_at_k(out, 10),
    'precision_at_50': precision_at_k(out, 50),
    'score_formula': {'visibility': 0.35, 'freshness': 0.25, 'position_opportunity': 0.20, 'low_ctr_opportunity': 0.15, 'depth_gap': 0.05},
    'csv_output': str(csv_path),
}
metrics_path.write_text(json.dumps(metrics, indent=2, sort_keys=True) + '\n')

print(f'Wrote working CSV: {csv_path}')
print(f'Wrote committed metrics receipt: {metrics_path}')
print(f'Base rate: {base_rate:.3f}')
print(f'Precision@10: {metrics["precision_at_10"]:.3f}')
print(f'Precision@50: {metrics["precision_at_50"]:.3f}')

Wrote working CSV: work/outputs/baseline_action_score.csv
Wrote committed metrics receipt: work/outputs/baseline_metrics.json
Base rate: 0.542
Precision@10: 0.400
Precision@50: 0.340


## 3. Top-10 review

For each item, show the action, reason code, confidence note, and what could make the recommendation wrong.

In [8]:
top10 = out.head(10).copy()
def confidence(row):
    if row['impressions_90d'] >= 3000 and row['reason_codes'] != 'general_review':
        return 'medium-high: visible demand plus explicit review signals'
    if row['impressions_90d'] >= 500 and row['reason_codes'] != 'general_review':
        return 'medium: inspect page context before acting'
    return 'low-medium: directional because demand is limited'

def failure_mode(row):
    if row['avg_position'] == 0:
        return 'No valid position data; position reasoning is unavailable.'
    if row['word_count'] == 0:
        return 'Word count is missing or unmeasured; thin-content reasoning may be incomplete.'
    return 'Topic, intent, competition, or client mix may explain the association.'

review = top10[['baseline_rank', 'content_id', 'suggested_action', 'reason_codes', 'baseline_action_score', 'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'word_count']].copy()
review['confidence_note'] = top10.apply(confidence, axis=1).to_numpy()
review['what_could_be_wrong'] = top10.apply(failure_mode, axis=1).to_numpy()
review_records = review.astype(object).where(pd.notna(review), None).to_dict(orient='records')
review_path.write_text(json.dumps(review_records, indent=2, default=str) + '\n')
print(f'Wrote committed top-10 receipt: {review_path}')
print(review.to_string(index=False))

Wrote committed top-10 receipt: work/outputs/baseline_top10_review.json
 baseline_rank           content_id       suggested_action                              reason_codes  baseline_action_score  impressions_90d  avg_position  ctr  days_since_last_update  word_count                                          confidence_note                                                            what_could_be_wrong
             1 content_5fe46e04994d refresh_and_review_ctr position_opportunity|low_ctr_visible_page               0.938801           517715           4.2 0.14                     104         0.0 medium-high: visible demand plus explicit review signals Word count is missing or unmeasured; thin-content reasoning may be incomplete.
             2 content_3430a8b94511 refresh_and_review_ctr position_opportunity|low_ctr_visible_page               0.938437           152617           3.3 0.29                     104         0.0 medium-high: visible demand plus explicit review signals Word count 

## 4. Weak picks + leakage check

Weak picks are high-ranked rows with limited evidence. Trend fields and the derived label are used only after ranking for evaluation; IDs are not score inputs.

In [9]:
top50 = out.head(50)
weak_mask = (top50['reason_codes'] == 'general_review') | (top50['impressions_90d'] < 250) | (top50['avg_position'] == 0)
weak_picks = top50.loc[weak_mask, ['baseline_rank', 'content_id', 'baseline_action_score', 'reason_codes', 'impressions_90d', 'avg_position', 'ctr', 'word_count']].head(10)
print('Weak-pick candidates:')
print(weak_picks.to_string(index=False) if len(weak_picks) else 'None found in the top 50.')

score_inputs = {'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'word_count', 'engagement_rate', 'sessions_90d'}
forbidden = {'trend_direction', 'trend_pct', 'is_declining_label', 'client_id', 'content_id'}
assert not (score_inputs & forbidden)
assert out['baseline_rank'].is_unique
leakage_receipt = {'score_inputs': sorted(score_inputs), 'excluded_fields': sorted(forbidden), 'status': 'PASS'}
leakage_path.write_text(json.dumps(leakage_receipt, indent=2, sort_keys=True) + '\n')
print(f'Wrote committed leakage receipt: {leakage_path}')
print('Leakage check: PASS')

Weak-pick candidates:
None found in the top 50.
Wrote committed leakage receipt: work/outputs/baseline_leakage_check.json
Leakage check: PASS


## 5. Self-check

- [x] The CSV queue is regenerated and not intended for git.
- [x] Metrics, top-10 review, and leakage receipts are written as JSON under `work/outputs/`.
- [x] Rule, reason codes, weak picks, and leakage checks are documented.
- [x] Claims remain observed, directional, and decision-support only.

In [10]:
checks = {
    'data_loaded': len(df) > 0,
    'csv_queue_written': csv_path.exists(),
    'metrics_json_written': metrics_path.exists(),
    'review_json_written': review_path.exists(),
    'leakage_json_written': leakage_path.exists(),
    'top_10_reviewed': len(review) == min(10, len(out)),
    'score_bounded': out['baseline_action_score'].between(0, 1).all(),
    'ranks_unique': out['baseline_rank'].is_unique,
}
for name, passed in checks.items():
    print(f'{name}: {passed}')
assert all(checks.values())
print('Self-check complete. CSV is disposable; JSON receipts are commit-worthy.')

data_loaded: True
csv_queue_written: True
metrics_json_written: True
review_json_written: True
leakage_json_written: True
top_10_reviewed: True
score_bounded: True
ranks_unique: True
Self-check complete. CSV is disposable; JSON receipts are commit-worthy.
